# Post-Simulation Analysis & Visualization

Loads results from CSVs generated by `run_simulations.ipynb` and produces plots and summary statistics.

**Prerequisite:** Run `run_simulations.ipynb` first to generate the CSV data files.

## Setup

In [ ]:
# Set working directory to project root
setwd("..")

library(ggplot2)
library(reshape2)
library(dplyr)
library(scales)

results_dir <- "results"

## Experiment 1: c_star Magnitude

In [ ]:
cat("=== Analyzing Experiment 1: c_star Magnitude ===\n")

exp1 <- read.csv(file.path(results_dir, "exp1_cstar_magnitude.csv"))

exp1_long <- melt(exp1[, c("multiplier", "reduction_diim", "reduction_pca")],
                  id.vars = "multiplier", variable.name = "method", value.name = "loss_reduction")

p1 <- ggplot(exp1_long, aes(x = multiplier, y = loss_reduction, color = method)) +
  geom_line(size = 1.2) + geom_point(size = 2.5) +
  scale_color_manual(values = c("reduction_diim" = "#3498DB", "reduction_pca" = "#2ECC71"),
    labels = c("DIIM Sectors", "PCA Sectors")) +
  labs(title = "Experiment 1: Loss Reduction vs c* Magnitude",
       x = "c* Multiplier", y = "Economic Loss Reduction", color = "Method") +
  theme_minimal() + theme(plot.title = element_text(face = "bold", size = 14), legend.position = "bottom")
print(p1)
ggsave(file.path(results_dir, "exp1_loss_reduction.png"), p1, width = 10, height = 6)

In [ ]:
exp1$advantage_pca <- exp1$reduction_pca - exp1$reduction_diim

p1b <- ggplot(exp1, aes(x = multiplier, y = advantage_pca)) +
  geom_line(size = 1.2, color = "#8E44AD") + geom_point(size = 2.5, color = "#8E44AD") +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red", size = 0.8) +
  annotate("rect", xmin = min(exp1$multiplier), xmax = max(exp1$multiplier),
           ymin = 0, ymax = Inf, fill = "#2ECC71", alpha = 0.1) +
  annotate("rect", xmin = min(exp1$multiplier), xmax = max(exp1$multiplier),
           ymin = -Inf, ymax = 0, fill = "#E74C3C", alpha = 0.1) +
  labs(title = "Experiment 1: PCA Advantage over DIIM",
       x = "c* Multiplier", y = "PCA - DIIM Loss Reduction") +
  theme_minimal() + theme(plot.title = element_text(face = "bold", size = 14))
print(p1b)
ggsave(file.path(results_dir, "exp1_pca_advantage.png"), p1b, width = 10, height = 6)

## Experiment 2: c_star Distribution

In [ ]:
cat("=== Analyzing Experiment 2: c_star Distribution ===\n")

exp2 <- read.csv(file.path(results_dir, "exp2_cstar_distribution.csv"))

exp2_long <- melt(exp2[, c("pattern", "reduction_diim", "reduction_pca", "c_star_gini")],
    id.vars = c("pattern", "c_star_gini"), variable.name = "method", value.name = "loss_reduction")
exp2_long$pattern <- reorder(exp2_long$pattern, exp2_long$c_star_gini)

p2 <- ggplot(exp2_long, aes(x = pattern, y = loss_reduction, fill = method)) +
  geom_bar(stat = "identity", position = "dodge", alpha = 0.85) +
  scale_fill_manual(values = c("reduction_diim" = "#3498DB", "reduction_pca" = "#2ECC71"),
    labels = c("DIIM Sectors", "PCA Sectors")) +
  labs(title = "Experiment 2: Loss Reduction by c* Distribution Pattern",
       x = "c* Distribution Pattern", y = "Economic Loss Reduction", fill = "Method") +
  theme_minimal() + theme(plot.title = element_text(face = "bold"), axis.text.x = element_text(angle = 45, hjust = 1), legend.position = "bottom")
print(p2)
ggsave(file.path(results_dir, "exp2_distribution_comparison.png"), p2, width = 10, height = 6)

## Experiment 3: q0 Uniformity

In [ ]:
cat("=== Analyzing Experiment 3: q0 Uniformity ===\n")

exp3 <- read.csv(file.path(results_dir, "exp3_q0_uniformity.csv"))
exp3$advantage_pca <- exp3$reduction_pca - exp3$reduction_diim

p3 <- ggplot(exp3, aes(x = factor(c_star_multiplier), y = advantage_pca, fill = q0_pattern)) +
  geom_bar(stat = "identity", position = "dodge", alpha = 0.85) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
  scale_fill_brewer(palette = "Set2", name = "q0 Pattern") +
  labs(title = "Experiment 3: PCA Advantage by q0 Pattern",
       x = "c* Multiplier", y = "PCA - DIIM Loss Reduction") +
  theme_minimal() + theme(plot.title = element_text(face = "bold"), legend.position = "bottom")
print(p3)
ggsave(file.path(results_dir, "exp3_q0_uniformity.png"), p3, width = 10, height = 6)

## Experiment 4: Lockdown × c_star Heatmap

In [ ]:
cat("=== Analyzing Experiment 4: Lockdown x c_star Grid ===\n")

exp4 <- read.csv(file.path(results_dir, "exp4_lockdown_cstar_grid.csv"))
exp4$advantage_pca <- exp4$reduction_pca - exp4$reduction_diim

p4 <- ggplot(exp4, aes(x = factor(c_star_multiplier), y = factor(lockdown_duration), fill = advantage_pca)) +
  geom_tile(color = "white", size = 0.3) +
  scale_fill_gradient2(low = "#E74C3C", mid = "white", high = "#2ECC71", midpoint = 0, name = "PCA\nAdvantage") +
  labs(title = "Experiment 4: PCA Advantage Heatmap",
       x = "c* Multiplier", y = "Lockdown Duration (days)") +
  theme_minimal() + theme(plot.title = element_text(face = "bold"), axis.text.x = element_text(angle = 45, hjust = 1))
print(p4)
ggsave(file.path(results_dir, "exp4_heatmap.png"), p4, width = 10, height = 8)

In [ ]:
p4b <- ggplot(exp4, aes(x = factor(c_star_multiplier), y = factor(lockdown_duration), fill = pca_wins)) +
  geom_tile(color = "white", size = 0.3) +
  scale_fill_manual(values = c("TRUE" = "#2ECC71", "FALSE" = "#E74C3C"),
    labels = c("TRUE" = "PCA Wins", "FALSE" = "DIIM Wins"), name = "Winner") +
  labs(title = "Experiment 4: Which Method Wins?",
       x = "c* Multiplier", y = "Lockdown Duration (days)") +
  theme_minimal() + theme(plot.title = element_text(face = "bold"), axis.text.x = element_text(angle = 45, hjust = 1))
print(p4b)
ggsave(file.path(results_dir, "exp4_binary_heatmap.png"), p4b, width = 10, height = 8)

## Experiment 5: Monte Carlo Analysis

In [ ]:
cat("=== Analyzing Experiment 5: Monte Carlo ===\n")

exp5 <- read.csv(file.path(results_dir, "exp5_monte_carlo.csv"))
cat(sprintf("  Overall PCA win rate: %.1f%% (%d / %d)\n",
    mean(exp5$pca_wins) * 100, sum(exp5$pca_wins), nrow(exp5)))

p5a <- ggplot(exp5, aes(x = c_star_gini, y = q0_gini, color = pca_wins)) +
  geom_point(alpha = 0.5, size = 2) +
  scale_color_manual(values = c("TRUE" = "#2ECC71", "FALSE" = "#E74C3C"),
    labels = c("TRUE" = "PCA Wins", "FALSE" = "DIIM Wins"), name = "Winner") +
  labs(title = "Experiment 5: Monte Carlo — Gini Coefficients",
       x = "c* Gini", y = "q0 Gini") +
  theme_minimal() + theme(plot.title = element_text(face = "bold"), legend.position = "bottom")
print(p5a)
ggsave(file.path(results_dir, "exp5_gini_scatter.png"), p5a, width = 10, height = 7)

In [ ]:
exp5$pca_wins_int <- as.integer(exp5$pca_wins)

logit_model <- glm(pca_wins_int ~ c_star_mean + c_star_sd + c_star_gini +
                     q0_mean + q0_sd + q0_gini +
                     cstar_q0_cor + c_star_pca_share + q0_pca_share +
                     lockdown_duration + overlap,
                   data = exp5, family = binomial)

coef_df <- data.frame(
  variable = names(coef(logit_model))[-1],
  coefficient = coef(logit_model)[-1],
  p_value = summary(logit_model)$coefficients[-1, 4]
)
coef_df$significant <- coef_df$p_value < 0.05
coef_df <- coef_df[order(abs(coef_df$coefficient), decreasing = TRUE), ]

p5c <- ggplot(coef_df, aes(x = reorder(variable, abs(coefficient)), y = coefficient, fill = significant)) +
  geom_bar(stat = "identity", alpha = 0.85) + geom_hline(yintercept = 0, linetype = "dashed") +
  scale_fill_manual(values = c("TRUE" = "#2ECC71", "FALSE" = "#BDC3C7"),
    labels = c("TRUE" = "p < 0.05", "FALSE" = "Not significant"), name = "Significance") +
  coord_flip() + labs(title = "Experiment 5: Logistic Regression Coefficients",
       x = "Feature", y = "Coefficient") +
  theme_minimal() + theme(plot.title = element_text(face = "bold"), legend.position = "bottom")
print(p5c)
ggsave(file.path(results_dir, "exp5_feature_importance.png"), p5c, width = 10, height = 7)

## Summary

In [ ]:
cat("\n══════════════════════════════════════════════════════\n")
cat("  SUMMARY OF KEY FINDINGS\n")
cat("══════════════════════════════════════════════════════\n\n")

pca_win_range_exp1 <- exp1$multiplier[exp1$pca_wins == TRUE]
if (length(pca_win_range_exp1) > 0) {
  cat(sprintf("Exp 1: PCA wins for c* multipliers: %s\n", paste(pca_win_range_exp1, collapse = ", ")))
} else { cat("Exp 1: PCA never wins\n") }

pca_win_patterns <- exp2$pattern[exp2$pca_wins == TRUE]
cat(sprintf("Exp 2: PCA wins for c* patterns: %s\n",
    if (length(pca_win_patterns) > 0) paste(pca_win_patterns, collapse = ", ") else "none"))

pca_win_pct_exp4 <- mean(exp4$pca_wins) * 100
cat(sprintf("Exp 4: PCA win rate across grid: %.1f%%\n", pca_win_pct_exp4))
cat(sprintf("Exp 5: Monte Carlo PCA win rate: %.1f%%\n", mean(exp5$pca_wins) * 100))